# CarePath DARAG - Part 1: Data Prep (CPU)

Runs on **Colab (CPU runtime) or any local machine** - the setup cell auto-detects.
The slow step is real Gipformer ASR over ViMedCSS to build `raw_asr -> gold_text`
pairs (paper Sec 3.1); it is CPU-bound, so on Colab use a **CPU runtime** to avoid
spending GPU units. Outputs are saved to Google Drive on Colab, or kept in the
repo's `artifacts/` locally - Part 2 picks them up either way.

In [ ]:
!pip install -q "datasets[audio]" soundfile huggingface_hub sherpa-onnx numpy jiwer

## 1. Set up the environment (Colab or local)

In [ ]:
# Set up the repo path + detect the runtime. Works on Colab AND a local machine.
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

def _find_repo(start: Path):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

REPO = _find_repo(Path.cwd())
if REPO is None and IN_COLAB:
    target = Path('/content/carepath')
    zip_path = os.environ.get('CAREPATH_REPO_ZIP', '/content/carepath.zip')
    repo_url = os.environ.get('CAREPATH_REPO_URL')
    if (target / 'pyproject.toml').exists():
        REPO = target
    elif Path(zip_path).exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall('/content/carepath_unzip')
        roots = [p.parent for p in Path('/content/carepath_unzip').rglob('pyproject.toml')]
        src = roots[0] if roots else Path('/content/carepath_unzip')
        target.mkdir(exist_ok=True)
        subprocess.run(f"cp -r '{src}'/* '{target}'/", shell=True, check=True)
        REPO = target
    elif repo_url:
        subprocess.run(['git', 'clone', repo_url, str(target)], check=True)
        REPO = target
    else:
        raise SystemExit('Colab: upload carepath.zip to /content, or set CAREPATH_REPO_ZIP / CAREPATH_REPO_URL.')

if REPO is None:
    raise SystemExit('Could not find the CarePath repo. Open this notebook from inside the cloned repo.')

os.chdir(REPO)
sys.path.insert(0, str(REPO / 'apps' / 'api'))
print(('Colab' if IN_COLAB else 'Local'), '| repo:', REPO)

In [ ]:
# Helper: run a pipeline CLI with PYTHONPATH set, streaming output, raising on failure.
import os, subprocess, sys

def run_step(args, env_extra=None):
    env = dict(os.environ)
    env["PYTHONPATH"] = "apps/api"
    env["PYTHONIOENCODING"] = "utf-8"
    if env_extra:
        env.update(env_extra)
    print(">>>", " ".join(args), flush=True)
    proc = subprocess.run([sys.executable, *args], env=env)
    if proc.returncode != 0:
        raise RuntimeError(f"step failed ({proc.returncode}): {' '.join(args)}")

In [ ]:
from carepath.gec.env import setup_backup
# Part 1 is CPU-only (Gipformer ASR), so no GPU is needed here.
BACKUP = setup_backup(IN_COLAB)

## 2. Run size - keep smoke defaults, raise for a real run

In [ ]:
LIMIT_PER_SPLIT = 20   # None for the full dataset
DATASET = 'tensorxt/ViMedCSS'
DATASTORE = 'artifacts/retrieval/term_datastore.json'
PAIRS = 'artifacts/gec_pairs/vimedcss_gipformer_pairs.jsonl'
print('LIMIT_PER_SPLIT =', LIMIT_PER_SPLIT)

## 3. Build the NE / code-switch datastore (paper Sec 4.2 Step 1)

In [ ]:
run_step([
    "scripts/gec/build_datastore.py",
    "--dataset", DATASET,
    "--limit-per-split", str(LIMIT_PER_SPLIT),
    "--output", DATASTORE,
])

## 4. Build real Gipformer GEC pairs (CPU - the long step)

`--resume` makes this restartable if the runtime drops.

In [ ]:
run_step([
    "scripts/gec/make_pairs.py",
    "--dataset", DATASET,
    "--output", PAIRS,
    "--limit-per-split", str(LIMIT_PER_SPLIT),
    "--datastore", DATASTORE,
    "--resume",
])

## 5. Quick baseline WER on the raw ASR (paper Table 1 style)

In [ ]:
run_step([
    "scripts/gec/evaluate.py",
    "--input", PAIRS,
    "--prediction-columns", "raw_asr",
    "--wer-output", "artifacts/evaluations/raw_baseline_wer.json",
    "--ne-f1-output", "artifacts/evaluations/raw_baseline_ne_f1.json",
])

## 6. Save artifacts (Drive on Colab, disk locally)

In [ ]:
from carepath.gec.env import save_artifacts
save_artifacts(BACKUP, [DATASTORE, PAIRS, 'artifacts/evaluations/raw_baseline_wer.json'])
print('Part 1 done. On Colab the files are on Drive; locally they are in artifacts/.')
print('Next: open Part 2 on a GPU runtime (Colab L4 or a local NVIDIA GPU).')